# AI-Powered Threat Intelligence

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import re
import time
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

class ThreatIntelligencePlatform:
    def __init__(self):
        self.anomaly_detector = IsolationForest(contamination=0.1, random_state=42)
        self.threat_classifier = GradientBoostingClassifier(n_estimators=100, random_state=42)
        self.behavioral_net = MLPClassifier(hidden_layer_sizes=(128, 64), random_state=42)
        self.scaler = StandardScaler()
        self.threat_graph = nx.DiGraph()
        self.ioc_database = {}
        self.attack_patterns = self.initialize_attack_patterns()

    def initialize_attack_patterns(self):
        return {
            'apt_patterns': [
                r'powershell.*-enc.*[A-Za-z0-9+/]{50,}',
                r'certutil.*-decode.*\.txt',
                r'reg.*add.*HKLM.*Run'
            ],
            'malware_patterns': [
                r'cmd.*\/c.*echo.*>.*\.bat',
                r'wscript.*\.vbs.*shell',
                r'rundll32.*javascript:'
            ],
            'lateral_movement': [
                r'net.*use.*\$.*password',
                r'psexec.*-s.*cmd',
                r'wmic.*process.*call.*create'
            ]
        }

    def generate_network_telemetry(self, n_samples=5000):
        np.random.seed(42)

        normal_data = {
            'src_ip': np.random.choice(['192.168.1.' + str(i) for i in range(1, 255)],
                                     size=int(n_samples * 0.85)),
            'dst_ip': np.random.choice(['10.0.0.' + str(i) for i in range(1, 255)],
                                     size=int(n_samples * 0.85)),
            'bytes_sent': np.random.lognormal(8, 1.5, int(n_samples * 0.85)),
            'packets': np.random.poisson(50, int(n_samples * 0.85)),
            'duration': np.random.exponential(30, int(n_samples * 0.85)),
            'ports': np.random.choice([80, 443, 22, 53, 25], int(n_samples * 0.85)),
            'protocol': np.random.choice(['TCP', 'UDP'], int(n_samples * 0.85), p=[0.7, 0.3]),
            'label': ['benign'] * int(n_samples * 0.85)
        }

        malicious_data = {
            'src_ip': np.random.choice(['192.168.1.' + str(i) for i in range(200, 255)],
                                     size=int(n_samples * 0.15)),
            'dst_ip': np.random.choice(['172.16.0.' + str(i) for i in range(1, 50)],
                                     size=int(n_samples * 0.15)),
            'bytes_sent': np.random.lognormal(12, 2, int(n_samples * 0.15)),
            'packets': np.random.poisson(200, int(n_samples * 0.15)),
            'duration': np.random.exponential(5, int(n_samples * 0.15)),
            'ports': np.random.choice([4444, 8080, 1337, 31337], int(n_samples * 0.15)),
            'protocol': np.random.choice(['TCP'], int(n_samples * 0.15)),
            'label': np.random.choice(['c2', 'exfiltration', 'scanning'], int(n_samples * 0.15))
        }

        combined_data = {}
        for key in normal_data:
            combined_data[key] = np.concatenate([normal_data[key], malicious_data[key]])

        df = pd.DataFrame(combined_data)
        return df.sample(frac=1).reset_index(drop=True)

    def extract_features(self, df):
        features = df.copy()

        features['src_ip_int'] = features['src_ip'].apply(lambda x: sum(int(i) << (8 * (3 - idx))
                                                                      for idx, i in enumerate(x.split('.'))))
        features['dst_ip_int'] = features['dst_ip'].apply(lambda x: sum(int(i) << (8 * (3 - idx))
                                                                      for idx, i in enumerate(x.split('.'))))
        features['bytes_per_packet'] = features['bytes_sent'] / (features['packets'] + 1)
        features['protocol_encoded'] = features['protocol'].map({'TCP': 1, 'UDP': 0})
        features['port_category'] = features['ports'].apply(self.categorize_port)

        return features[['src_ip_int', 'dst_ip_int', 'bytes_sent', 'packets',
                        'duration', 'bytes_per_packet', 'protocol_encoded', 'port_category']]

    def categorize_port(self, port):
        if port in [80, 443, 8080, 8443]:
            return 1
        elif port in [22, 23, 3389]:
            return 2
        elif port in [53, 67, 68]:
            return 3
        elif port > 1024:
            return 4
        else:
            return 0

    def train_models(self, df):
        X = self.extract_features(df)
        y = (df['label'] != 'benign').astype(int)
        y_multi = pd.get_dummies(df['label']).values

        X_scaled = self.scaler.fit_transform(X)

        self.anomaly_detector.fit(X_scaled)
        self.threat_classifier.fit(X_scaled, y)
        self.behavioral_net.fit(X_scaled, y_multi)

        anomaly_scores = self.anomaly_detector.decision_function(X_scaled)
        return anomaly_scores

    def detect_apt_campaign(self, log_data):
        apt_indicators = []

        for pattern_type, patterns in self.attack_patterns.items():
            for pattern in patterns:
                matches = [re.search(pattern, log, re.IGNORECASE) for log in log_data if re.search(pattern, log, re.IGNORECASE)]
                if matches:
                    apt_indicators.append({
                        'type': pattern_type,
                        'pattern': pattern,
                        'matches': len(matches),
                        'severity': self.calculate_severity(pattern_type, len(matches))
                    })

        return apt_indicators

    def calculate_severity(self, pattern_type, match_count):
        base_scores = {'apt_patterns': 8, 'malware_patterns': 6, 'lateral_movement': 9}
        base_score = base_scores.get(pattern_type, 5)

        multiplier = min(1 + (match_count - 1) * 0.1, 2.0)
        return min(base_score * multiplier, 10)

    def build_attack_graph(self, threat_data):
        self.threat_graph.clear()

        for _, row in threat_data.iterrows():
            if row['label'] != 'benign':
                self.threat_graph.add_edge(row['src_ip'], row['dst_ip'],
                                         weight=row['bytes_sent'],
                                         threat_type=row['label'])

        centrality = nx.betweenness_centrality(self.threat_graph)
        pagerank = nx.pagerank(self.threat_graph)

        return centrality, pagerank

    def generate_iocs(self, threat_data):
        iocs = {'ips': set(), 'domains': set(), 'file_hashes': set(), 'urls': set()}

        malicious_data = threat_data[threat_data['label'] != 'benign']

        iocs['ips'].update(malicious_data['src_ip'].unique())
        iocs['ips'].update(malicious_data['dst_ip'].unique())

        for _ in range(20):
            fake_hash = hashlib.sha256(f"malware_{np.random.randint(1000000)}".encode()).hexdigest()
            iocs['file_hashes'].add(fake_hash)

        suspicious_domains = [f"malware-{i}.badsite.com" for i in range(10)]
        iocs['domains'].update(suspicious_domains)

        return iocs

    def threat_hunting(self, df, hunt_queries):
        results = {}

        for query_name, conditions in hunt_queries.items():
            matches = df.copy()

            for condition in conditions:
                field, operator, value = condition
                if operator == '>':
                    matches = matches[matches[field] > value]
                elif operator == '<':
                    matches = matches[matches[field] < value]
                elif operator == '==':
                    matches = matches[matches[field] == value]
                elif operator == 'contains':
                    matches = matches[matches[field].astype(str).str.contains(str(value))]

            results[query_name] = len(matches)

        return results

    def behavioral_clustering(self, df):
        X = self.extract_features(df)
        X_scaled = self.scaler.transform(X)

        clustering = DBSCAN(eps=0.5, min_samples=5)
        clusters = clustering.fit_predict(X_scaled)

        cluster_analysis = {}
        for cluster_id in set(clusters):
            if cluster_id != -1:
                cluster_data = df[clusters == cluster_id]
                cluster_analysis[cluster_id] = {
                    'size': len(cluster_data),
                    'threat_ratio': (cluster_data['label'] != 'benign').mean(),
                    'dominant_threat': cluster_data['label'].mode().iloc[0] if len(cluster_data) > 0 else 'unknown'
                }

        return clusters, cluster_analysis

    def visualize_threat_landscape(self, df):
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))

        threat_counts = df['label'].value_counts()
        axes[0, 0].pie(threat_counts.values, labels=threat_counts.index, autopct='%1.1f%%')
        axes[0, 0].set_title('Threat Distribution')

        port_threat = pd.crosstab(df['ports'], df['label'])
        port_threat.plot(kind='bar', stacked=True, ax=axes[0, 1])
        axes[0, 1].set_title('Threats by Port')
        axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

        hourly_threats = np.random.poisson(10, 24)
        axes[1, 0].plot(range(24), hourly_threats, 'ro-')
        axes[1, 0].set_title('Threat Activity by Hour')
        axes[1, 0].set_xlabel('Hour of Day')
        axes[1, 0].set_ylabel('Threat Count')

        if len(self.threat_graph.nodes()) > 0:
            pos = nx.spring_layout(self.threat_graph, k=2, iterations=50)
            nx.draw(self.threat_graph, pos, ax=axes[1, 1],
                   node_size=100, alpha=0.7, arrows=True,
                   node_color='red', edge_color='gray')
            axes[1, 1].set_title('Threat Network Graph')

        plt.tight_layout()
        plt.show()

platform = ThreatIntelligencePlatform()

telemetry_data = platform.generate_network_telemetry(10000)
print(f"Generated {len(telemetry_data)} telemetry records")
print(f"Threat distribution:\n{telemetry_data['label'].value_counts()}")

anomaly_scores = platform.train_models(telemetry_data)
print(f"Anomaly detection trained. Score range: [{np.min(anomaly_scores):.3f}, {np.max(anomaly_scores):.3f}]")

sample_logs = [
    "powershell -enc UG93ZXJTaGVsbCAtV2luZG93U3R5bGUgSGlkZGVu",
    "certutil -decode malware.txt malware.exe",
    "reg add HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run /v Malware",
    "cmd /c echo @echo off > backdoor.bat",
    "net use \\\\target\\c$ password123 /user:admin"
]

apt_detection = platform.detect_apt_campaign(sample_logs)
print(f"APT indicators detected: {len(apt_detection)}")

centrality, pagerank = platform.build_attack_graph(telemetry_data)
print(f"Attack graph built with {len(platform.threat_graph.nodes())} nodes")

iocs = platform.generate_iocs(telemetry_data)
print(f"Generated IOCs - IPs: {len(iocs['ips'])}, Hashes: {len(iocs['file_hashes'])}")

hunt_queries = {
    'high_volume_transfer': [('bytes_sent', '>', 1000000)],
    'suspicious_ports': [('ports', '==', 4444)],
    'rapid_connections': [('duration', '<', 1)]
}

hunt_results = platform.threat_hunting(telemetry_data, hunt_queries)
print(f"Threat hunting results: {hunt_results}")

clusters, cluster_analysis = platform.behavioral_clustering(telemetry_data)
print(f"Behavioral clustering: {len(set(clusters)) - (1 if -1 in clusters else 0)} clusters")

platform.visualize_threat_landscape(telemetry_data)